<a href="https://colab.research.google.com/github/MR-just01/Llama3.2-Reasoning/blob/main/04_llama_lora_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!git clone https://github.com/MR-just01/Llama3.2-Reasoning

Cloning into 'Llama3.2-Reasoning'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 104 (delta 56), reused 31 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 21.45 MiB | 10.16 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [14]:
%cd Llama3.2-Reasoning

/content/Llama3.2-Reasoning


In [16]:
import pandas as pd

df = pd.read_csv("data/processed/reasoning_dataset (1).csv")
print(df.shape)
df.head()

(30193, 6)


,instruction,input,reasoning,answer,dataset,task_type
0,Solve the following math reasoning problem ste...,Nicole collected 400 Pokemon cards. Cindy coll...,Cindy has 400 x 2 = <<400*2=800>>800 cards.\nN...,150,gsm8k,math_reasoning
1,Solve the following math reasoning problem ste...,James had two browsers on his computer. In eac...,The total number of tabs in three windows of e...,60,gsm8k,math_reasoning
2,Solve the following multiple-choice math reaso...,The 100-milliliter solution of sugar and water...,In the original solution the amount of sugar i...,50,AQUA-RAT,math_reasoning
3,Solve the following multiple-choice math reaso...,"If 0.75 : x :: 5 : 8, then x is equal to:\n\nC...",Explanation:\n(x x 5) = (0.75 x 8)\nx=6/5\n=1....,1.2,AQUA-RAT,math_reasoning
4,Solve the following multiple-choice math reaso...,The price of Darjeeling tea (in rupees per kil...,Explanation :\nPrice of Darjeeling tea (in rup...,May 20,AQUA-RAT,math_reasoning


In [1]:
!pip install -q -U \
transformers \
accelerate \
bitsandbytes>=0.46.1 \
peft \
trl \
datasets \
sentencepiece

In [17]:
import transformers
import trl
import peft
import torch

print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

Transformers: 5.14.1
TRL: 1.9.2
PEFT: 0.20.0
Torch: 2.11.0+cpu


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [18]:
def format_prompt(row):

    messages = [
        {
            "role": "user",
            "content":
                f"{row['instruction']}\n\n{row['input']}"
        },
        {
            "role": "assistant",
            "content":
                f"Reasoning:\n{row['reasoning']}\n\n"
                f"Final Answer:\n{row['answer']}"
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [19]:
df["text"] = df.apply(format_prompt, axis=1)

In [20]:
print(df["text"].iloc[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 05 Aug 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Solve the following math reasoning problem step by step.

Nicole collected 400 Pokemon cards. Cindy collected twice as many, and Rex collected half of Nicole and Cindy's combined total. If Rex divided his card equally among himself and his three younger siblings, how many cards does Rex have left?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Reasoning:
Cindy has 400 x 2 = <<400*2=800>>800 cards.
Nicole and Cindy have 400 + 800 = <<400+800=1200>>1200 cards.
Rex has 1200/2 = <<1200/2=600>>600 cards.
Rex is left with 600/(3+1=4) = <<600/4=150>>150 cards

Final Answer:
150<|eot_id|>


In [21]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

In [22]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df,
    preserve_index=False
)

In [23]:
print(train_dataset)
print(val_dataset)

Dataset({
    features: ['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type', 'text'],
    num_rows: 27173
})
Dataset({
    features: ['instruction', 'input', 'reasoning', 'answer', 'dataset', 'task_type', 'text'],
    num_rows: 3020
})


In [26]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

# **k-bit training**

In [27]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [28]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

## **Configure LORA**

In [29]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [30]:
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [31]:
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [32]:
from trl import SFTConfig

training_args = SFTConfig(
    # Output
    output_dir="./llama3_reasoning",

    # Training
    num_train_epochs=1,
    learning_rate=2e-4,

    # Batching
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    # Precision
    fp16=True,
    bf16=False,

    # Optimizer
    optim="paged_adamw_8bit",

    # Scheduler
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # Logging
    logging_steps=10,
    report_to="none",

    # Checkpoints
    save_strategy="epoch",
    eval_strategy="epoch",

    # Dataset
    dataset_text_field="text",
    max_length=1024,
    packing=False,

    # Misc
    remove_unused_columns=False,
    seed=42,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [33]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

Adding EOS to train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/27173 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/3020 [00:00<?, ? examples/s]

In [35]:
import math

print("="*40)

print("Training Samples:",
      len(train_dataset))

print("Validation Samples:",
      len(val_dataset))

steps = math.ceil(
    len(train_dataset) /
    (
        training_args.per_device_train_batch_size *
        training_args.gradient_accumulation_steps
    )
)

print("Steps per Epoch:",
      steps)

print("="*40)

token_lengths = df["text"].apply(
    lambda x: len(
        tokenizer(x)["input_ids"]
    )
)

print(token_lengths.describe())

print("Maximum Tokens:",
      token_lengths.max())

Training Samples: 27173
Validation Samples: 3020
Steps per Epoch: 3397
count    30193.000000
mean       205.107210
std         65.741242
min         80.000000
25%        162.000000
50%        195.000000
75%        237.000000
max        857.000000
Name: text, dtype: float64
Maximum Tokens: 857


In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.device_count())